# Southern Ocean, 1/12°, MARBL BGC

Everything south of ~30°S, cut out of the global tx1_12 grid. The slice keeps all 360° of
longitude, so MOM6 runs it zonally reentrant. Antarctica closes the south edge, and the only
open boundary is 30°S.

Needs CrocoDash `southern-ocean-bgc` with submodules updated.

## Grid
Cut the global supergrid with `subgrid_from_supergrid`. A box spanning all 360° keeps every
column, so the subgrid wraps around. It loads the global grid first (~1 min, ~16 GB).

In [ ]:
from pathlib import Path
from CrocoDash.grid import Grid

CROC = "/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/croc"
GLOBAL_HGRID = f"{CROC}/crocogallerydata/grids/ocean_hgrid_trimmed.nc"   # tx1_12
GLOBAL_TOPO = "/glade/work/gmarques/cesm/tx1_12/topography/ocean_topog_tx1_12v1_260213.nc"

name = "so_tx1_12"

grid = Grid.subgrid_from_supergrid(
    path=GLOBAL_HGRID,
    llc=(-90.0, -287.0),   # (lat, lon): below the grid's south edge, at its west seam
    urc=(-30.0, 73.0),     # 360 deg of longitude -> keeps every column
    name=name,
)
print(grid.nx, grid.ny, "cyclic:", grid.supergrid.is_cyclic_x)

## Topography
Gustavo Marques' global tx1_12 topography; `from_topo_file` finds the subgrid inside it.

In [ ]:
from CrocoDash.topo import Topo

topo = Topo.from_topo_file(grid, GLOBAL_TOPO, min_depth=9.5)
topo.depth.plot()

## Vertical grid

In [ ]:
from CrocoDash.vgrid import VGrid

vgrid = VGrid.hyperbolic(nk=75, depth=6000.0, ratio=20.0, name=name)

## CESM case
JRA atmosphere, CICE sea ice, MOM6 with MARBL, GLOFAS runoff.

In [ ]:
from CrocoDash.case import Case

casename = name
cesmroot = Path("~/work/crocodile2026/CESM").expanduser()
inputdir = Path("/glade/u/home/manishrv/scratch/croc_input") / casename
caseroot = Path("~/croc_cases").expanduser() / casename

case = Case(
    cesmroot=cesmroot,
    caseroot=caseroot,
    inputdir=inputdir,
    ocn_grid=grid,
    ocn_vgrid=vgrid,
    ocn_topo=topo,
    project="ncgd0011",
    machine="derecho",
    ntasks_ocn=3840,
    override=False,
    compset="1850_DATM%JRA_SLND_CICE_MOM6%REGIONAL%MARBL-BIO_DROF%GLOFAS_SGLC_SWAV",
)

## Forcings
CESM-HR FOSI_BGC (0.1° POP, physics + MARBL tracers) for the IC and the 30°S boundary.

In [ ]:
case.configure_forcings(
    date_range=["2000-01-01 00:00:00", "2000-02-01 00:00:00"],
    product_name="CESM_POP_OUTPUT",
    function_name="get_cesm_single_variable_data",
    marbl_ic_filepath=f"{CROC}/crocogallerydata/bgc/ecosys_jan_IC_omip_latlon_1x1_180W_c231221.nc",
    rof_esmf_mesh_filepath=f"{CROC}/rof/glofas/dis24/GLOFAS_esmf_mesh_v4.nc",
    global_river_nutrients_filepath=f"{CROC}/rof/river_nutrients/river_nutrients.GNEWS_GNM.glofas.20250916.64bit.nc",
)
print(case.boundaries)

In [ ]:
case.process_forcings()